# 01 Data Collection and Preprocessing

This notebook collects research papers from arXiv and Semantic Scholar, cleans and chunks the text, creates train/validation/test splits, and prepares an instruction-style dataset for PEFT fine-tuning.

In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / 'src').exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'src').exists():
    raise RuntimeError('Could not locate project root containing src/')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.paper_search import collect_papers_for_topic
from src.preprocessing import preprocess_papers, split_dataset, build_finetune_dataset, split_records, save_jsonl

DATA_DIR = PROJECT_ROOT / 'data'
DATA_DIR.mkdir(parents=True, exist_ok=True)

RAW_CSV = DATA_DIR / 'raw_papers.csv'
PROCESSED_CSV = DATA_DIR / 'processed_papers.csv'
TRAIN_CSV = DATA_DIR / 'train.csv'
VAL_CSV = DATA_DIR / 'val.csv'
TEST_CSV = DATA_DIR / 'test.csv'
FINETUNE_JSONL = DATA_DIR / 'finetune_dataset.jsonl'
FINETUNE_TRAIN_JSONL = DATA_DIR / 'finetune_train.jsonl'
FINETUNE_VAL_JSONL = DATA_DIR / 'finetune_val.jsonl'
FINETUNE_TEST_JSONL = DATA_DIR / 'finetune_test.jsonl'

pd.set_option('display.max_colwidth', 120)
print(f'Project root: {PROJECT_ROOT}')

Project root: /Users/Lenovo/Desktop/sem 6/Gen_AI Project


## Topic List

These are the narrowed research topics for initial paper collection.

In [2]:
topics = [
    'Open-source large language models',
    'Hugging Face transformer model ecosystem',
    'Instruction-tuned open-source language models',
    'Parameter-efficient fine-tuning for open-source LLMs',
    'LoRA fine-tuning for transformer language models',
    'Large language models for scientific text understanding',
    'Large language models for literature review generation',
    'Large language models for question answering over research papers',
    'LLM hallucination detection and mitigation',
    'Hallucination mitigation in retrieval augmented generation',
    'Research assistant chatbots using large language models',
    'AI chatbots for education using large language models',
    'Retrieval-augmented generation for research assistants',
    'Retrieval-augmented generation for academic question answering',
    'RAG for literature review synthesis',
    'RAG for citation-grounded text generation',
    'Evaluation of retrieval-augmented generation systems',
    'Transformers for scientific document summarization',
    'Long-context transformers for document understanding',
    'Semantic search for research paper exploration',
    'Neural information retrieval for scholarly search',
    'Academic paper recommendation using embeddings',
    'Multi-agent systems for literature review generation',
    'Multi-agent RAG architectures for knowledge intensive tasks'
]

len(topics)

24

## Data Collection

Set the per-source limits and optionally provide a Semantic Scholar API key. The pipeline will still work with arXiv alone if Semantic Scholar requests fail.

In [3]:
ARXIV_LIMIT = 25
SEMANTIC_LIMIT = 25
SEMANTIC_API_KEY = None  # Replace with your key if you have one.

all_raw_dfs = []
collection_log = []

for topic in topics:
    print(f'Collecting papers for: {topic}')
    topic_df = collect_papers_for_topic(
        topic,
        arxiv_limit=ARXIV_LIMIT,
        semantic_limit=SEMANTIC_LIMIT,
        semantic_api_key=SEMANTIC_API_KEY,
    )
    topic_count = len(topic_df)
    collection_log.append({'topic': topic, 'papers_collected': topic_count})
    if topic_count > 0:
        all_raw_dfs.append(topic_df)

collection_log_df = pd.DataFrame(collection_log)
collection_log_df

,topic,papers_collected
0,Open-source large language models,25
1,Hugging Face transformer model ecosystem,25
2,Instruction-tuned open-source language models,25
3,Parameter-efficient fine-tuning for open-source LLMs,25
4,LoRA fine-tuning for transformer language models,25
5,Large language models for scientific text understanding,25
6,Large language models for literature review generation,25
7,Large language models for question answering over research papers,25
8,LLM hallucination detection and mitigation,25
9,Hallucination mitigation in retrieval augmented generation,25


In [4]:
if not all_raw_dfs:
    raise ValueError('No papers were collected. Check your internet connection, topics, or API limits.')

raw_df = pd.concat(all_raw_dfs, ignore_index=True)
raw_df = raw_df.drop_duplicates(subset=['title', 'abstract']).reset_index(drop=True)
raw_df.to_csv(RAW_CSV, index=False)

print(f'Total unique raw papers collected: {len(raw_df)}')
raw_df.head(10)

Total unique raw papers collected: 448


,paper_id,title,abstract,authors,year,venue,url,topic,source
0,2409.11272v7,LOLA -- An Open-Source Massively Multilingual Large Language Model,"This paper presents LOLA, a massively multilingual large language model trained on more than 160 languages using a s...","Nikit Srivastava, Denis Kuchelev, Tatiana Moteu Ngoli, Kshitij Shetty, Michael Röder, Hamada Zahera, Diego Moussalle...",2024,arXiv,http://arxiv.org/abs/2409.11272v7,Open-source large language models,arxiv
1,2501.05032v2,Enhancing Human-Like Responses in Large Language Models,This paper explores the advancements in making large language models (LLMs) more human-like. We focus on techniques ...,"Ethem Yağız Çalık, Talha Rüzgar Akkuş",2025,arXiv,http://arxiv.org/abs/2501.05032v2,Open-source large language models,arxiv
2,2503.16581v1,Investigating Retrieval-Augmented Generation in Quranic Studies: A Study of 13 Open-Source Large Language Models,Accurate and contextually faithful responses are critical when applying large language models (LLMs) to sensitive an...,"Zahra Khalila, Arbi Haza Nasution, Winda Monika, Aytug Onan, Yohei Murakami, Yasir Bin Ismail Radi, Noor Mohammad Os...",2025,arXiv,http://arxiv.org/abs/2503.16581v1,Open-source large language models,arxiv
3,2402.14679v2,Is Self-knowledge and Action Consistent or Not: Investigating Large Language Model's Personality,"In this study, we delve into the validity of conventional personality questionnaires in capturing the human-like per...","Yiming Ai, Zhiwei He, Ziyin Zhang, Wenhong Zhu, Hongkun Hao, Kai Yu, Lingjun Chen, Rui Wang",2024,arXiv,http://arxiv.org/abs/2402.14679v2,Open-source large language models,arxiv
4,2309.13734v2,Prompting and Fine-Tuning Open-Sourced Large Language Models for Stance Classification,"Stance classification, the task of predicting the viewpoint of an author on a subject of interest, has long been a f...","Iain J. Cruickshank, Lynnette Hui Xian Ng",2023,arXiv,http://arxiv.org/abs/2309.13734v2,Open-source large language models,arxiv
5,2405.11357v3,Large Language Models Lack Understanding of Character Composition of Words,Large language models (LLMs) have demonstrated remarkable performances on a wide range of natural language tasks. Ye...,"Andrew Shin, Kunitake Kaneko",2024,arXiv,http://arxiv.org/abs/2405.11357v3,Open-source large language models,arxiv
6,2403.09676v1,Unmasking the Shadows of AI: Investigating Deceptive Capabilities in Large Language Models,"This research critically navigates the intricate landscape of AI deception, concentrating on deceptive behaviours of...",Linge Guo,2024,arXiv,http://arxiv.org/abs/2403.09676v1,Open-source large language models,arxiv
7,2310.00905v2,All Languages Matter: On the Multilingual Safety of Large Language Models,"Safety lies at the core of developing and deploying large language models (LLMs). However, previous safety benchmark...","Wenxuan Wang, Zhaopeng Tu, Chang Chen, Youliang Yuan, Jen-tse Huang, Wenxiang Jiao, Michael R. Lyu",2023,arXiv,http://arxiv.org/abs/2310.00905v2,Open-source large language models,arxiv
8,2204.06745v1,GPT-NeoX-20B: An Open-Source Autoregressive Language Model,"We introduce GPT-NeoX-20B, a 20 billion parameter autoregressive language model trained on the Pile, whose weights w...","Sid Black, Stella Biderman, Eric Hallahan, Quentin Anthony, Leo Gao, Laurence Golding, Horace He, Connor Leahy, Kyle...",2022,arXiv,http://arxiv.org/abs/2204.06745v1,Open-source large language models,arxiv
9,2407.01505v1,Self-Cognition in Large Language Models: An Exploratory Study,"While Large Language Models (LLMs) have achieved remarkable success across various applications, they also raise con...","Dongping Chen, Jiawen Shi, Yao Wan, Pan Zhou, Neil Zhenqiang Gong, Lichao Sun",2024,arXiv,http://arxiv.org/abs/2407.01505v1,Open-source large language models,arxiv


## Dataset Inspection

In [5]:
summary_df = pd.DataFrame(
    {
        'total_papers': [len(raw_df)],
        'unique_topics': [raw_df['topic'].nunique()],
        'missing_abstracts': [raw_df['abstract'].isna().sum()],
        'missing_titles': [raw_df['title'].isna().sum()],
    }
)
summary_df

,total_papers,unique_topics,missing_abstracts,missing_titles
0,448,24,0,0


## Preprocessing and Chunking

In [6]:
processed_df = preprocess_papers(raw_df)
processed_df.to_csv(PROCESSED_CSV, index=False)

print(f'Total processed chunks: {len(processed_df)}')
processed_df[['paper_id', 'title', 'chunk_id', 'chunk_text', 'topic']].head(10)

Total processed chunks: 928


,paper_id,title,chunk_id,chunk_text,topic
0,2409.11272v7,LOLA -- An Open-Source Massively Multilingual Large Language Model,2409.11272v7_chunk_0,"LOLA -- An Open-Source Massively Multilingual Large Language Model. This paper presents LOLA, a massively multilingu...",Open-source large language models
1,2409.11272v7,LOLA -- An Open-Source Massively Multilingual Large Language Model,2409.11272v7_chunk_1,"of the datasets, and a balanced exploration of the model's strengths and limitations. As an open-source model, LOLA ...",Open-source large language models
2,2501.05032v2,Enhancing Human-Like Responses in Large Language Models,2501.05032v2_chunk_0,Enhancing Human-Like Responses in Large Language Models. This paper explores the advancements in making large langua...,Open-source large language models
3,2503.16581v1,Investigating Retrieval-Augmented Generation in Quranic Studies: A Study of 13 Open-Source Large Language Models,2503.16581v1_chunk_0,Investigating Retrieval-Augmented Generation in Quranic Studies: A Study of 13 Open-Source Large Language Models. Ac...,Open-source large language models
4,2503.16581v1,Investigating Retrieval-Augmented Generation in Quranic Studies: A Study of 13 Open-Source Large Language Models,2503.16581v1_chunk_1,"small (e.g., Llama3.2:3b, Phi3:3.8b). A Retrieval-Augmented Generation (RAG) is used to make up for the problems tha...",Open-source large language models
5,2503.16581v1,Investigating Retrieval-Augmented Generation in Quranic Studies: A Study of 13 Open-Source Large Language Models,2503.16581v1_chunk_2,"very well on faithfulness (4.619) and relevance (4.857), showing the promise of smaller architectures that have been...",Open-source large language models
6,2402.14679v2,Is Self-knowledge and Action Consistent or Not: Investigating Large Language Model's Personality,2402.14679v2_chunk_0,"Is Self-knowledge and Action Consistent or Not: Investigating Large Language Model's Personality. In this study, we ...",Open-source large language models
7,2309.13734v2,Prompting and Fine-Tuning Open-Sourced Large Language Models for Stance Classification,2309.13734v2_chunk_0,"Prompting and Fine-Tuning Open-Sourced Large Language Models for Stance Classification. Stance classification, the t...",Open-source large language models
8,2309.13734v2,Prompting and Fine-Tuning Open-Sourced Large Language Models for Stance Classification,2309.13734v2_chunk_1,reduce or even eliminate the need for manual annotations. We investigate 10 open-source models and 7 prompting schem...,Open-source large language models
9,2405.11357v3,Large Language Models Lack Understanding of Character Composition of Words,2405.11357v3_chunk_0,Large Language Models Lack Understanding of Character Composition of Words. Large language models (LLMs) have demons...,Open-source large language models


## Train / Validation / Test Split

In [7]:
train_df, val_df, test_df = split_dataset(processed_df)

train_df.to_csv(TRAIN_CSV, index=False)
val_df.to_csv(VAL_CSV, index=False)
test_df.to_csv(TEST_CSV, index=False)

split_summary = pd.DataFrame(
    {
        'split': ['train', 'val', 'test'],
        'rows': [len(train_df), len(val_df), len(test_df)],
    }
)
split_summary

,split,rows
0,train,649
1,val,139
2,test,140


## Fine-Tuning Dataset Preparation

In [8]:
finetune_records = build_finetune_dataset(raw_df)
finetune_train, finetune_val, finetune_test = split_records(finetune_records)

save_jsonl(finetune_records, FINETUNE_JSONL)
save_jsonl(finetune_train, FINETUNE_TRAIN_JSONL)
save_jsonl(finetune_val, FINETUNE_VAL_JSONL)
save_jsonl(finetune_test, FINETUNE_TEST_JSONL)

print(f'Fine-tuning samples created: {len(finetune_records)}')
print(f'Train/Val/Test: {len(finetune_train)} / {len(finetune_val)} / {len(finetune_test)}')
pd.DataFrame(finetune_records).groupby('task').size().reset_index(name='examples')

Fine-tuning samples created: 1566
Train/Val/Test: 1252 / 157 / 157


,task,examples
0,comparative_analysis,89
1,evidence_based_qa,433
2,literature_review,89
3,paper_summary,433
4,research_gap_analysis,89
5,technical_explanation,433


## Final Output Check

In [9]:
generated_files = [
    RAW_CSV,
    PROCESSED_CSV,
    TRAIN_CSV,
    VAL_CSV,
    TEST_CSV,
    FINETUNE_JSONL,
    FINETUNE_TRAIN_JSONL,
    FINETUNE_VAL_JSONL,
    FINETUNE_TEST_JSONL,
]
pd.DataFrame(
    {
        'file': [str(path.name) for path in generated_files],
        'exists': [path.exists() for path in generated_files],
        'size_bytes': [path.stat().st_size if path.exists() else 0 for path in generated_files],
    }
)

,file,exists,size_bytes
0,raw_papers.csv,True,666672
1,processed_papers.csv,True,3457792
2,train.csv,True,2435990
3,val.csv,True,514053
4,test.csv,True,507931
5,finetune_dataset.jsonl,True,3228643
6,finetune_train.jsonl,True,2573378
7,finetune_val.jsonl,True,327360
8,finetune_test.jsonl,True,327905
